# XLKD05 Dense Mapped LogitKD: Next-phase entry

Notebook CPU này khóa lộ trình tiếp theo sau khi tái lập Dense Mapped
Logit-KD trên ba seed `42, 123, 2026`.

Mục tiêu:

1. xác minh gói tổng hợp ba seed và audit độc lập;
2. ghi nhận rõ gate tái lập đã đạt nhưng chưa mở Held-out Test;
3. khóa đối chứng ngân sách huấn luyện tương đương;
4. khóa paired bootstrap theo image ID;
5. chỉ sau hai bước trên mới cho phép sàng lọc Feature KD seed 42.

Notebook không huấn luyện, không đọc ảnh và không dùng Test.

## Kaggle inputs bắt buộc

- output aggregate ba seed;
- output audit được tạo bởi
  `XLKD05_Dense_Mapped_LogitKD_Output_Audit.py`.


## Cell 1 — Khám phá và xác minh input


In [1]:
import csv
import hashlib
import json
import os
import platform
import time
from datetime import datetime, timezone
from pathlib import Path

STARTED = time.perf_counter()
INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
# Search the complete input tree. Kaggle may expose notebook outputs
# below /kaggle/input/<dataset>, while a local workspace may itself
# contain a generic folder named "notebooks" that is unrelated.
NOTEBOOK_ROOT = INPUT_ROOT
OUTPUT_ROOT = Path(os.environ.get(
    "KLCN_OUTPUT_ROOT",
    "/kaggle/working//"
    "XLKD05_Dense_Mapped_LogitKD_next_phase",
))
AUDIT_DIR = OUTPUT_ROOT / "audits"
CONFIG_DIR = OUTPUT_ROOT / "configs"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
for directory in [OUTPUT_ROOT, AUDIT_DIR, CONFIG_DIR, MANIFEST_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

def now():
    return datetime.now(timezone.utc).isoformat()

def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(
            lambda: handle.read(8 * 1024 * 1024), b""
        ):
            digest.update(block)
    return digest.hexdigest()

def exactly_one(name):
    matches = sorted(NOTEBOOK_ROOT.rglob(name))
    assert len(matches) == 1, (
        f"Cần đúng một {name}; tìm thấy {len(matches)}: {matches}"
    )
    return matches[0].resolve()

AGGREGATE_PATH = exactly_one(
    "XLKD06_DENSE_MAPPED_LOGIT_KD_"
    "THREE_SEED_AGGREGATE_COMPLETE.json"
)
OUTPUT_AUDIT_PATH = exactly_one(
    "XLKD05_Dense_Mapped_LogitKD_OUTPUT_AUDIT.json"
)
AGGREGATE = read_json(AGGREGATE_PATH)
OUTPUT_AUDIT = read_json(OUTPUT_AUDIT_PATH)

assert AGGREGATE["status"] == "complete"
assert AGGREGATE["seeds"] == [42, 123, 2026]
assert AGGREGATE["source_runs_verified"] == "3 / 3"
assert AGGREGATE["protocol_consistency"] == "exact"
assert AGGREGATE["reproduction_status"] == "passed"
assert AGGREGATE["cider_gate"] == "3 / 3"
assert AGGREGATE["test_set_status"] == "sealed"
assert AGGREGATE["test_set_used"] is False

assert OUTPUT_AUDIT["status"] == (
    "passed_with_scientific_follow_up_required"
)
assert OUTPUT_AUDIT["gates"]["reproduction_gate_passed"] is True
assert OUTPUT_AUDIT["gates"][
    "final_scientific_claim_gate_passed"
] is False
assert OUTPUT_AUDIT["decision"]["current"] == (
    "REPRODUCTION_GATE_PASSED"
)

write_json(AUDIT_DIR / "cell01_input_integrity.json", {
    "schema_version": "xlkd05_next_phase_cell01_v1",
    "created_at_utc": now(),
    "status": "passed",
    "aggregate_sha256": sha256(AGGREGATE_PATH),
    "output_audit_sha256": sha256(OUTPUT_AUDIT_PATH),
    "seeds": [42, 123, 2026],
    "source_runs_verified": "3 / 3",
    "protocol_consistency": "exact",
    "python": platform.python_version(),
    "training_started": False,
    "image_payload_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
})

print("Input integrity            : PASSED")
print("Reproduction gate         : PASSED")
print("Final scientific gate     : PENDING")
print("Test Set status           : SEALED")


Input integrity            : PASSED
Reproduction gate         : PASSED
Final scientific gate     : PENDING
Test Set status           : SEALED


## Cell 2 — Khóa đối chứng ngân sách tương đương


In [2]:
MATCHED_BUDGET = {
    "schema_version": "xlkd05_matched_budget_contract_v1",
    "created_at_utc": now(),
    "status": "locked",
    "purpose": (
        "Separate the effect of KD from the extra post-baseline "
        "optimization budget."
    ),
    "seeds": [42, 123, 2026],
    "initialization": (
        "The exact S1 baseline-best checkpoint for the same seed."
    ),
    "arms": {
        "control": {
            "loss": "human CE only",
            "alpha_human_ce": 1.0,
            "alpha_kd": 0.0,
        },
        "treatment": {
            "loss": "dense mapped Logit-KD",
            "alpha_human_ce": 0.5,
            "alpha_kd": 0.5,
            "temperature": 4.0,
        },
    },
    "must_match": [
        "train/restval image IDs",
        "initial checkpoint SHA-256",
        "optimizer family and parameter groups",
        "learning-rate schedule",
        "physical batch and gradient accumulation",
        "maximum optimizer-step budget",
        "validation frequency and decoding",
        "checkpoint selection by validation CIDEr-D",
    ],
    "required_exports_per_seed": [
        "best checkpoint and SHA-256",
        "epoch history CSV",
        "5,000 validation predictions keyed by image_id",
        "COCO caption metrics",
        "optimizer-step count and wall-clock time",
    ],
    "selection_split": "validation",
    "test_set_status": "sealed",
    "test_set_used": False,
}
MATCHED_BUDGET_PATH = (
    CONFIG_DIR / "matched_budget_control_contract.json"
)
write_json(MATCHED_BUDGET_PATH, MATCHED_BUDGET)

write_json(AUDIT_DIR / "cell02_matched_budget_contract.json", {
    "schema_version": "xlkd05_next_phase_cell02_v1",
    "created_at_utc": now(),
    "status": "passed",
    "contract_sha256": sha256(MATCHED_BUDGET_PATH),
    "control_arms_required": 3,
    "test_set_status": "sealed",
    "test_set_used": False,
})

print("Matched-budget contract   : LOCKED")
print("Required CE controls      : 3 seeds")
print("Selection split           : Validation")
print("Test Set status           : SEALED")


Matched-budget contract   : LOCKED
Required CE controls      : 3 seeds
Selection split           : Validation
Test Set status           : SEALED


## Cell 3 — Khóa paired bootstrap theo ảnh


In [3]:
BOOTSTRAP = {
    "schema_version": "xlkd05_paired_bootstrap_contract_v1",
    "created_at_utc": now(),
    "status": "locked",
    "unit": "image_id",
    "validation_image_count": 5000,
    "pairing": (
        "Control and treatment must contain the exact same sorted "
        "image_id set and the same reference captions."
    ),
    "replicates": 10000,
    "bootstrap_seed": 20261018,
    "confidence_level": 0.95,
    "interval": "percentile paired bootstrap",
    "primary_metric": "CIDEr-D",
    "secondary_metrics": [
        "BLEU-1", "BLEU-4", "METEOR", "ROUGE-L"
    ],
    "success_gate": {
        "mean_delta_cider_d_greater_than": 0.0,
        "lower_95_ci_delta_cider_d_greater_than": 0.0,
        "required_seed_count": "at least 2 / 3",
        "no_serious_secondary_metric_regression": True,
    },
    "test_set_status": "sealed",
    "test_set_used": False,
}
BOOTSTRAP_PATH = CONFIG_DIR / "paired_bootstrap_contract.json"
write_json(BOOTSTRAP_PATH, BOOTSTRAP)

write_json(AUDIT_DIR / "cell03_bootstrap_contract.json", {
    "schema_version": "xlkd05_next_phase_cell03_v1",
    "created_at_utc": now(),
    "status": "passed",
    "contract_sha256": sha256(BOOTSTRAP_PATH),
    "image_level_pairing_required": True,
    "held_out_test_may_be_opened": False,
    "test_set_status": "sealed",
    "test_set_used": False,
})

print("Paired bootstrap contract : LOCKED")
print("Replicates                : 10000")
print("Primary metric            : CIDEr-D")
print("Held-out Test             : MUST REMAIN SEALED")


Paired bootstrap contract : LOCKED
Replicates                : 10000
Primary metric            : CIDEr-D
Held-out Test             : MUST REMAIN SEALED


## Cell 4 — Khóa Feature KD preflight


In [4]:
FEATURE_KD = {
    "schema_version": "xlkd05_feature_kd_preflight_contract_v1",
    "created_at_utc": now(),
    "status": "locked",
    "prerequisite": (
        "Run only as a validation-only screening branch; do not use "
        "it to bypass matched-budget and bootstrap requirements."
    ),
    "teacher": {
        "name": "ExpansionNet v2",
        "frozen": True,
        "feature_tap": "input_linear(swin_transf(image))",
        "expected_shape": ["B", 144, 512],
    },
    "student": {
        "name": "MobileNetV3-Large S1",
        "feature_tap": "visual memory",
        "expected_shape": ["B", 49, 256],
    },
    "alignment": {
        "spatial": "adaptive average pooling 12x12 to 7x7",
        "adapter": "Linear(512,256) + LayerNorm",
        "adapter_trainable": True,
        "identity_projection_forbidden": True,
    },
    "loss": {
        "primary": "L2-normalized MSE",
        "secondary_preregistered": "cosine distance",
        "lambda_primary": 0.1,
        "teacher_stop_gradient": True,
    },
    "preflight_gates": {
        "weighted_feature_to_caption_gradient_ratio": [0.05, 0.50],
        "finite_loss_and_gradients": True,
        "deterministic_repeat": True,
        "t4_peak_memory_mib_less_than": 14500,
        "optimizer_steps": 0,
    },
    "screening": {
        "seed": 42,
        "selection_split": "validation",
        "confirmation_seeds": [123, 2026],
        "confirmation_only_after_seed42_gate": True,
    },
    "test_set_status": "sealed",
    "test_set_used": False,
}
FEATURE_KD_PATH = CONFIG_DIR / "feature_kd_preflight_contract.json"
write_json(FEATURE_KD_PATH, FEATURE_KD)

write_json(
    OUTPUT_ROOT / "FEATURE_KD_GPU_PREFLIGHT_AUTHORIZED.json",
    {
        "schema_version": (
            "xlkd05_feature_kd_gpu_preflight_authorized_v1"
        ),
        "created_at_utc": now(),
        "status": "authorized",
        "authorized_next": (
            "Feature-tap, adapter, gradient-ratio and memory "
            "preflight on a fixed Train batch"
        ),
        "contract": {
            "relative_path": (
                FEATURE_KD_PATH.relative_to(OUTPUT_ROOT).as_posix()
            ),
            "sha256": sha256(FEATURE_KD_PATH),
        },
        "allowed_seed": 42,
        "full_training_authorized": False,
        "optimizer_steps_allowed": 0,
        "held_out_test_may_be_opened": False,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

write_json(AUDIT_DIR / "cell04_feature_kd_contract.json", {
    "schema_version": "xlkd05_next_phase_cell04_v1",
    "created_at_utc": now(),
    "status": "passed",
    "contract_sha256": sha256(FEATURE_KD_PATH),
    "screening_seed": 42,
    "full_training_authorized": False,
    "test_set_status": "sealed",
    "test_set_used": False,
})

print("Feature KD contract       : LOCKED")
print("GPU preflight             : AUTHORIZED")
print("Full training             : NOT AUTHORIZED")
print("Test Set status           : SEALED")


Feature KD contract       : LOCKED
GPU preflight             : AUTHORIZED
Full training             : NOT AUTHORIZED
Test Set status           : SEALED


## Cell 5 — Đóng gói và lộ trình thực thi


In [5]:
artifacts = []
for path in sorted(OUTPUT_ROOT.rglob("*")):
    if not path.is_file() or path.is_relative_to(MANIFEST_DIR):
        continue
    artifacts.append({
        "relative_path": path.relative_to(OUTPUT_ROOT).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256(path),
    })

MANIFEST_PATH = MANIFEST_DIR / "next_phase_manifest.csv"
with MANIFEST_PATH.open(
    "w", encoding="utf-8", newline=""
) as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["relative_path", "size_bytes", "sha256"],
    )
    writer.writeheader()
    writer.writerows(artifacts)

MARKER_PATH = OUTPUT_ROOT / (
    "XLKD05_Dense_Mapped_LogitKD_"
    "NEXT_PHASE_ENTRY_COMPLETE.json"
)
write_json(MARKER_PATH, {
    "schema_version": "xlkd05_next_phase_entry_complete_v1",
    "created_at_utc": now(),
    "status": "complete",
    "reproduction_gate": "passed",
    "final_scientific_gate": "pending",
    "matched_budget_contract": {
        "relative_path": (
            MATCHED_BUDGET_PATH.relative_to(OUTPUT_ROOT).as_posix()
        ),
        "sha256": sha256(MATCHED_BUDGET_PATH),
    },
    "paired_bootstrap_contract": {
        "relative_path": (
            BOOTSTRAP_PATH.relative_to(OUTPUT_ROOT).as_posix()
        ),
        "sha256": sha256(BOOTSTRAP_PATH),
    },
    "feature_kd_preflight_contract": {
        "relative_path": (
            FEATURE_KD_PATH.relative_to(OUTPUT_ROOT).as_posix()
        ),
        "sha256": sha256(FEATURE_KD_PATH),
    },
    "next_order": [
        "Run three CE-only matched-budget controls.",
        "Run paired image-level bootstrap on Validation.",
        "Run Feature KD GPU compatibility preflight (zero updates).",
        "If the preflight passes, screen Feature KD on seed 42.",
        "Confirm only the winning method on seeds 123 and 2026.",
        "Lock final method before opening Held-out Test once.",
    ],
    "manifest": {
        "relative_path": MANIFEST_PATH.relative_to(
            OUTPUT_ROOT
        ).as_posix(),
        "sha256": sha256(MANIFEST_PATH),
        "rows": len(artifacts),
    },
    "training_started": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})

print("=" * 78)
print("XLKD05 — NEXT-PHASE ENTRY")
print("=" * 78)
print("Status                    : COMPLETE")
print("Reproduction gate         : PASSED")
print("Final scientific gate     : PENDING")
print("Matched-budget controls   : REQUIRED")
print("Paired bootstrap          : REQUIRED")
print("Feature KD GPU preflight  : AUTHORIZED / ZERO UPDATES")
print("Held-out Test             : SEALED")
print("Marker                    :", MARKER_PATH)


XLKD05 — NEXT-PHASE ENTRY
Status                    : COMPLETE
Reproduction gate         : PASSED
Final scientific gate     : PENDING
Matched-budget controls   : REQUIRED
Paired bootstrap          : REQUIRED
Feature KD GPU preflight  : AUTHORIZED / ZERO UPDATES
Held-out Test             : SEALED
Marker                    : /kaggle/working//XLKD05_Dense_Mapped_LogitKD_next_phase/XLKD05_Dense_Mapped_LogitKD_NEXT_PHASE_ENTRY_COMPLETE.json
